In [ ]:
import plotly.graph_objects as go
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
import numpy as np

In [ ]:
ticker = 'MSTR'
start_date = '2025-01-01'
end_date = '2026-06-13'
df = yf.download(ticker, start=start_date, end = end_date)
df.to_csv('Stock.csv')

In [ ]:
# --- Define Split Ratio ---
train_ratio = 0.8  # 80% for training, 20% for testing
total_days = len(df)
split_idx = int(total_days * train_ratio)

# --- Split the Data Chronologically ---
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

print(f"Total Rows: {total_days}")
print(f"Training Rows (First 80%): {len(train_df)} (From {train_df.index[0].date()} to {train_df.index[-1].date()})")
print(f"Testing Rows (Last 20%): {len(test_df)} (From {test_df.index[0].date()} to {test_df.index[-1].date()})")

# --- Extract Features Separately ---
train_features = train_df[['Open', 'Close']].to_numpy()
test_features = test_df[['Open', 'Close']].to_numpy()

In [ ]:
# 1. Fit the scaler on TRAINING data only
train_min = train_features.min(axis=0)
train_max = train_features.max(axis=0)

# 2. Scale the training windows
T = 60
window_data_train = train_features[0:T, :]
scaled_window_train = (window_data_train - train_min) / (train_max - train_min)
inputVar_train = scaled_window_train.T  # Shape: (2, 60)

# 3. Scale the testing windows using TRAINING parameters
window_data_test = test_features[0:T, :]
scaled_window_test = (window_data_test - train_min) / (train_max - train_min)
inputVar_test = scaled_window_test.T  # Shape: (2, 60)

In [ ]:
# Input-to-Hidden weights == (3, 2)
WxH = np.array([
    [ 0.5, -0.2],
    [ 0.1,  0.8],
    [-0.4,  0.3]
])  

# Hidden-to-Hidden weights == (3, 3)
WhH = np.array([
    [ 0.9,  0.1, -0.2],
    [ 0.0,  0.8,  0.3],
    [-0.1,  0.2,  0.7]
])  

# Hidden-to-Output weights

WhY = np.array([
    [ 0.4, -0.5,  0.2],
    [-0.3,  0.7,  0.1]
])  

# Biases
hS0 = np.zeros((3, 1))  
hS0_train = np.zeros((3,1))
biasWV = np.zeros((3, 1))
biasWY = np.zeros((2, 1))

In [ ]:
# tanh (from scratch), with clipping to avoid exp() overflow on large inputs
def tanh(x):
    x = np.clip(x, -30, 30)
    return (np.exp(x) - np.exp(-x)) / (np.exp(x) + np.exp(-x))

In [ ]:
hidden_states_train = []
outputs_train = []
hS_current = hS0_train.copy()

for t in range(T):
    xt = inputVar_train[:, t].reshape(2, 1)
    hState_train = tanh(np.dot(WxH, xt) + np.dot(WhH, hS_current) + biasWV)
    outputVal_train = np.dot(WhY, hState_train) + biasWY
    
    # Track historical metrics for BPTT step
    hidden_states_train.append(hState_train)
    outputs_train.append(outputVal_train)
    
    hS_current = hState_train

# Final Prediction for next step after training window
tomorrow_hidden_train = tanh(np.dot(WhH, hS_current) + biasWV)
pred_scaled_train = np.dot(WhY, tomorrow_hidden_train) + biasWY
pred_unscaled_train = pred_scaled_train.flatten() * (train_max - train_min) + train_min

In [ ]:
#BPTT
targets_train = []
for t in range(T):
    next_idx = min(t + 1, T - 1)
    targets_train.append(inputVar_train[:, next_idx].reshape(2, 1))

dLdWxh = np.zeros_like(WxH)
dLdWhh = np.zeros_like(WhH)
dLdWhy = np.zeros_like(WhY)

hiddenStateLossAccumulator_next = np.zeros((3,1))

for t in reversed(range(T)):

    dLossdyt = 2 * (outputs_train[t] - targets_train[t])
    hiddenStateLossAccumulatorStep1 = (np.dot(WhY.T,dLossdyt) + np.dot(WhH.T, hiddenStateLossAccumulator_next))
    hiddenStateLossAccumulatorStep2 = (1-((hidden_states_train[t])**2))
    hiddenStateLossAcumulator = hiddenStateLossAccumulatorStep1 * hiddenStateLossAccumulatorStep2
    
    xt = inputVar_train[:, t].reshape(2, 1)
    dLdWxh += np.dot(hiddenStateLossAcumulator, xt.T)
    prev_h = hidden_states_train[t - 1] if t > 0 else hS0_train
    dLdWhh += np.dot(hiddenStateLossAcumulator, prev_h.T)
    dLdWhy += np.dot(dLossdyt, hidden_states_train[t].T)
    
    dLdbWV += hiddenStateLossAcumulator
    dLdbWY += dLossdyt
    hiddenStateLossAccumulator_next = hiddenStateLossAcumulator
    
# Assuming a learning rate of 0.005
learning_rate = 0.005

# The Final Weight Update
WhY -= learning_rate * dLdWhy
WhH -= learning_rate * dLdWhh
WxH -= learning_rate * dLdWxh

biasWV -= learning_rate * dLdbWV
biasWY -= learning_rate * dLdbWY

In [ ]:
hS0_test = np.zeros((3, 1))  # Reset hidden state for testing separation

for t in range(T):
    xt = inputVar_test[:, t].reshape(2, 1)
    hState_test = tanh(np.dot(WxH, xt) + np.dot(WhH, hS0_test) + biasWV)
    outputVal_test = np.dot(WhY, hState_test) + biasWY
    hS0_test = hState_test

# Final Prediction for next step after testing window
tomorrow_hidden_test = tanh(np.dot(WhH, hS0_test) + biasWV)
pred_scaled_test = np.dot(WhY, tomorrow_hidden_test) + biasWY
pred_unscaled_test = pred_scaled_test.flatten() * (train_max - train_min) + train_min
print(f"Prediction right after Test Window: {pred_unscaled_test}")